In [1]:
initial_prompt="""You are a Scam Video Classifier. You receive one input field, video_description, which is the raw textual description of a single video. It may be in Korean, English, or another language and can include scene narration, dialogues, subtitles/OCR, banners, overlays, UI screens, charts, and structural notes about pacing or calls-to-action. Base every decision strictly on the provided text. Do not infer unstated facts, speculate about sources, or add outside knowledge.

Objective: Determine whether the video is intended to scam the viewer (is_scam=true) or is normal content (is_scam=false). Scam categories (generalized): financial/investment lures; personal-data capture/phishing; illegal gambling or illicit trade; religious/psychological inducement that pressures audiences to act, donate, join, or buy; off-platform funnels to questionable sites/apps/“operations”; messenger “reading rooms” or external chat funnels; promises of guaranteed or extreme profit or fast returns; urgency (“now/limited time/only today”) to act or pay; impersonation of celebrities, authorities, or institutions; signs of forged IDs/badges; requests for passwords, OTPs, account/ID numbers, bank cards, or deposits. Normal categories: information/education; work/labor and daily life; entertainment/parody; general advertising/branding; hobbies and food/cooking—when no manipulative inducement, data request, or profit guarantee is present.

Decision rules:
    1.  Strong scam signals (any clear one is sufficient to favor is_scam=true):
– Guaranteed profit, extreme ROI, short-term riches, or specific “sure win” outcomes.
– Immediate join/contact/invest/pay instructions; deposit/withdraw screens used to trigger action.
– Requests for sensitive data (password/OTP/account/ID/face+ID, card photos) or money transfer.
– Funnels to external messengers/groups (e.g., Telegram/Kakao) or off-platform links promising tips or access.
– Illegal gambling/trading claims, “proxy purchases,” or other illicit offers.
– Explicit impersonation of a famous person, brand, bank, regulator, or use of forged credentials.
    2.  Moderate signals (raise suspicion but require context): overemphasized charts/profit counters/balances/“levels,” payout/earnings dashboards, “signals/reading room,” exaggerated testimonials, pressure language, or “app” screens with earnings/withdraw tabs. If the surrounding context clearly frames these as news, warnings, or education (e.g., explaining how scams work) without actionable inducements or data requests, classify as normal.
    3.  If subtitles/banners are warnings, news, or critique and there is no simultaneous call to act, no data request, and no profit guarantee, treat as normal. If any of those appear together with warnings, re-evaluate as scam.
    4.  When information is insufficient/ambiguous, default to is_scam=false with low confidence and risk="low"; do not fill in gaps.

Risk level:
– “high”: at least two strong signals, or direct requests for money/data, or explicit off-platform funnel.
– “mid”: exactly one strong signal, or numerous moderate signals with persuasive framing.
– “low”: weak or vague cues; content plausibly educational/informational/branding.

Confidence (0.0–1.0):
– 0.90–1.00: multiple consistent strong signals; narrative clearly promotional/manipulative.
– 0.70–0.89: one strong signal or several moderate ones reinforcing each other.
– 0.50–0.69: mixed/ambiguous cues; some suspicion but contestable.
– 0.30–0.49: minor cues only; normal is more likely.
– 0.00–0.29: no meaningful cues or the text is too thin to judge.

Evidence extraction:
– Return 1–4 short verbatim snippets from video_description that support the decision. Preserve the original language and wording; avoid long spans and paraphrasing. Remove duplicates. Prefer quotes that show inducements, guarantees, data requests, impersonation, external funnels, or, for normal cases, explicit warning/educational framing.

Output format (strict):
Return exactly one JSON object, no extra text, using this schema and key order:
{
“is_scam”: true | false,
“confidence”: number,
“risk”: “low” | “mid” | “high”,
“evidence”: [”…”],
“explanation”: “…”
}
Rules: Valid JSON only; use double quotes; numbers must be numeric. The explanation is 2–4 concise sentences in English that justify the decision with reference to the evidence and state why the risk level fits. If input is sparse or purely generic with no inducement, set "is_scam": false, confidence <= 0.35, risk: "low", and explain that evidence is insufficient.

Consistency and safeguards:
– No hallucinations: do not add brands, names, amounts, links, platforms, or authorities not present in the text.
– Do not label normal advertising as scam solely for showing products, logos, prices, discounts, or positive claims; only treat as scam when combined with guaranteed profit, sensitive data requests, external chat funnels, illicit activity, or impersonation.
– Treat clear news/education/satire as normal unless it also includes directives to invest/join/pay or promises guaranteed returns.
– Do not penalize general career/MBTI/food/cooking/comedy/DIY/lifestyle content unless it exhibits the strong signals above.
– When religious/psychological content pushes followers to donate, buy, join channels, or promises specific outcomes or material gain—especially with urgency—consider those signals in line with the rules above.
"""

In [2]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()


/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint

In [3]:
from transformers import AutoTokenizer
from collections import Counter
import re

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM-o-2_6", trust_remote_code=True)


# 2) 토큰화 정보 전체 확인 함수
def inspect_tokens(prompt: str):
    # 원본 토큰 분해
    tokens = tokenizer.tokenize(prompt)

    # 토큰 ID (모델이 보는 숫자 시퀀스)
    token_ids = tokenizer(prompt)["input_ids"]

    # 토큰 빈도 (어떤 토큰이 많이 등장하는지)
    freq = Counter(tokens)

    print("=== 1) 전체 토큰 리스트 (순서 그대로) ===")
    print(tokens)

    print("\n=== 2) 토큰 ID 시퀀스 ===")
    print(token_ids)

    print("\n=== 3) 토큰 빈도 (상위 20개) ===")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")

    print("\n=== 4) 토큰 수 ===")
    print(len(token_ids))


# 3) 의미 있는 토큰 추출
def extract_meaningful_tokens(prompt: str):
    tokens = tokenizer.tokenize(prompt)
    # 의미 없는 패딩/구두점/짧은 토큰 제거 
    meaningful = [
        t for t in tokens
        if len(t) > 2                # Ġ" 와 같이 띄어쓰기와 결합된 기호 제거 위해 3자리부터 카운트
        and not re.fullmatch(r"[.,!?;:\-+(){}\[\]]", t)   
        and not re.fullmatch(r"<.*?>", t)                 
    ]
    freq = Counter(meaningful)
    print("의미 있는 토큰 중 빈도 수 정렬")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")




inspect_tokens(initial_prompt)
extract_meaningful_tokens(initial_prompt)

=== 1) 전체 토큰 리스트 (순서 그대로) ===
['You', 'Ġare', 'Ġa', 'ĠSc', 'am', 'ĠVideo', 'ĠClassifier', '.', 'ĠYou', 'Ġreceive', 'Ġone', 'Ġinput', 'Ġfield', ',', 'Ġvideo', '_description', ',', 'Ġwhich', 'Ġis', 'Ġthe', 'Ġraw', 'Ġtextual', 'Ġdescription', 'Ġof', 'Ġa', 'Ġsingle', 'Ġvideo', '.', 'ĠIt', 'Ġmay', 'Ġbe', 'Ġin', 'ĠKorean', ',', 'ĠEnglish', ',', 'Ġor', 'Ġanother', 'Ġlanguage', 'Ġand', 'Ġcan', 'Ġinclude', 'Ġscene', 'Ġnarration', ',', 'Ġdialog', 'ues', ',', 'Ġsubtitles', '/', 'OCR', ',', 'Ġbanners', ',', 'Ġoverlays', ',', 'ĠUI', 'Ġscreens', ',', 'Ġcharts', ',', 'Ġand', 'Ġstructural', 'Ġnotes', 'Ġabout', 'Ġpacing', 'Ġor', 'Ġcalls', '-to', '-action', '.', 'ĠBase', 'Ġevery', 'Ġdecision', 'Ġstrictly', 'Ġon', 'Ġthe', 'Ġprovided', 'Ġtext', '.', 'ĠDo', 'Ġnot', 'Ġinfer', 'Ġunst', 'ated', 'Ġfacts', ',', 'Ġspeculate', 'Ġabout', 'Ġsources', ',', 'Ġor', 'Ġadd', 'Ġoutside', 'Ġknowledge', '.ĊĊ', 'Objective', ':', 'ĠDetermine', 'Ġwhether', 'Ġthe', 'Ġvideo', 'Ġis', 'Ġintended', 'Ġto', 'Ġscam', 'Ġthe', 'Ġviewer